In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
import joblib

df = pd.read_csv("../data/processed/intelliprice_featured.csv")

# ============================================================
# 1. FEATURE SEÇİMİ
# ============================================================
features = [
    'Monthly Premium Auto',
    'Months Since Last Claim',
    'Months Since Policy Inception',
    'Number of Open Complaints',
    'Number of Policies',
    'Income',
    'city_risk_score',
    'parts_inflation_index',
    'no_claim_score',
    'income_risk_ratio_log',
    'location_score',
    'Coverage',
    'Vehicle Class',
    'Vehicle Size',
    'EmploymentStatus',
    'Marital Status',
    'Gender'
]

target = 'Total Claim Amount'

# ============================================================
# 2. KATEGORİK KOLONLARI ENCODE ET
# ============================================================
cat_cols = ['Coverage', 'Vehicle Class', 'Vehicle Size', 
            'EmploymentStatus', 'Marital Status', 'Gender']

df_model = df[features + [target]].copy()
df_model[cat_cols] = df_model[cat_cols].astype('category')

# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ============================================================
# 4. LİGHTGBM MODELİ (TWEEDİE)
# ============================================================
model = lgb.LGBMRegressor(
    objective='tweedie',
    tweedie_variance_power=1.5,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=cat_cols
)

# ============================================================
# 5. DEĞERLENDİRME
# ============================================================
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n📊 Model Performansı:")
print(f"MAE  (Ortalama Mutlak Hata) : {mae:.2f} TL")
print(f"RMSE (Karekök Ortalama Hata): {rmse:.2f} TL")
print(f"\nOrtalama Gerçek Hasar : {y_test.mean():.2f} TL")
print(f"Ortalama Tahmin       : {y_pred.mean():.2f} TL")

# ============================================================
# 6. MODELİ KAYDET
# ============================================================
joblib.dump(model, "../models/intelliprice_model.pkl")
print("\n✅ Model kaydedildi!")

Train: (7307, 17), Test: (1827, 17)


c:\Users\tunah\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\tunah\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")



📊 Model Performansı:
MAE  (Ortalama Mutlak Hata) : 73.49 TL
RMSE (Karekök Ortalama Hata): 116.26 TL

Ortalama Gerçek Hasar : 421.81 TL
Ortalama Tahmin       : 419.62 TL

✅ Model kaydedildi!


In [2]:
pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.0/1.5 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 3.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
